# ECG Signal Processing — Colab Friendly Notebook

This notebook is prepared for Google Colab. It installs required packages, attempts to clone the repository, and runs a self-contained synthetic example demonstrating filtering, R-peak detection, BPM calculation, QRS analysis, and figure output. If you already have the repository files in Colab, you can skip the cloning step.

In [1]:
# Install dependencies (run once).
%pip install --quiet wfdb neurokit2 matplotlib scipy numpy

ERROR: Operation cancelled by user
^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Clone the repository into the Colab environment (if public)
!git clone https://github.com/kubascikmichal/ECG-signal-processing.git || true
import sys
# Make the repository root importable so `from src import ...` works
sys.path.insert(0, '/content/ECG-signal-processing')
print('Added to sys.path:', sys.path[0])

In [ ]:
# Run the synthetic example using the package modules
from src import filters, rpeak_detection, bpm_analysis, qrs_analysis, visualization
from IPython.display import Image, display
import numpy as np
import matplotlib.pyplot as plt

fs = 360.0
duration_s = 15.0
n_samples = int(fs * duration_s)
t = np.arange(n_samples) / fs
beat_interval_s = 0.8
beat_samples = np.arange(0, n_samples, int(beat_interval_s * fs))
signal = np.zeros(n_samples, dtype=float)
signal[beat_samples] = 1.0
qrs_width_s = 0.03
kernel_radius = int(0.06 * fs)
kernel_x = np.linspace(-kernel_radius, kernel_radius, 2 * kernel_radius + 1)
kernel = np.exp(-0.5 * (kernel_x / (qrs_width_s * fs)) ** 2)
kernel = kernel / np.sum(kernel)
synthetic = np.convolve(signal, kernel, mode='same')
synthetic += 0.08 * np.sin(2 * np.pi * 0.33 * t)
synthetic += 0.02 * np.random.randn(n_samples)

filtered = filters.bandpass_filter(synthetic, fs, lowcut=0.5, highcut=40.0)
r_peaks, n_peaks = rpeak_detection.detect_r_peaks(filtered, fs)
rr_intervals = bpm_analysis.calculate_rr_intervals(r_peaks, fs)
bpm = bpm_analysis.calculate_bpm(rr_intervals)
avg_bpm = bpm_analysis.calculate_average_bpm(rr_intervals)

print(f'Found peaks: {n_peaks}')
print(f'Average BPM (instantaneous mean): {avg_bpm:.1f}')

qrs_summary = qrs_analysis.analyze_qrs(filtered, r_peaks, fs)
print('Beat count:', qrs_summary['beat_count'])
print('Average R-wave amplitude:', qrs_summary['average_r_wave_amplitude'])
print('Estimated QRS duration (ms):', qrs_summary['estimated_qrs_duration_ms'])

# Save and display figure (Colab will render the PNG)
out_path = visualization.plot_r_peaks(filtered, r_peaks, fs, output_dir='outputs/figures', record_id='colab_synthetic')
display(Image(str(out_path)))

## Notes
- If the repository is private, replace the clone cell with an upload of the repository files or mount your Google Drive and copy files into `/content/`.
- The NeuroKit2-based detectors are optional; the code falls back to SciPy if NeuroKit2 is unavailable or fails.